In [1]:
import hashlib
import random
import math
import heapq
import time

In [1]:
import hashlib
import datetime
import time
import json
import random
import math
import heapq
import inspect
from collections import deque,defaultdict
import heapq
from math import sqrt

############################################
# Helper Function for Input Conversion
############################################

def prompt_for_parameter(param_name: str, param_type: type):
    """
    Prompts the user to enter a value for a given parameter and converts it to the desired type.
    Note: For list and dict, the conversion follows the original behavior.
    """
    raw_value = input(f"Enter {param_name}: ").strip()
    try:
        if param_type == int:
            return int(raw_value)
        elif param_type == float:
            return float(raw_value)
        elif param_type == list:
            # This will create a list of characters from the input.
            return list(raw_value)
        elif param_type == dict:
            # This converts the input string to a dict (may not work as expected if input is not in proper format)
            return dict(raw_value)
        else:
            return raw_value
    except Exception as e:
        print(f"Error converting input for {param_name}: {e}")
        return raw_value

def sim_prompt_for_parameter(param_name: str, param_type: type):
    """
    Prompts the user to enter a value for a given parameter and converts it to the desired type.
    Note: For list and dict, the conversion follows the original behavior.
    """
    try:
        if param_type == int:
            return 0
        elif param_type == float:
            return 0.0
        elif param_type == list:
            # This will create a list of characters from the input.
            return []
        elif param_type == dict:
            # This converts the input string to a dict (may not work as expected if input is not in proper format)
            return {}
        else:
            return 'Name'
    except Exception as e:
        print(f"Error converting input for {param_name}: {e}")
        return raw_value

############################################
# Smart Instance: Instantiator Class
############################################

class Instantiator:
    """
    Dynamically loads classes from a provided script, instantiates them,
    lists their members, runs member functions, and creates a composite class
    that inherits from all loaded classes.
    """
    
    def __init__(self, script: str):
        """
        Executes the provided script in an isolated namespace and extracts class definitions.
        
        Args:
            script (str): A string containing Python class definitions.
        """
        self.namespace = {}
        exec(script, self.namespace)
        self.classes = {name: obj for name, obj in self.namespace.items() if isinstance(obj, type)}
    
    def get_class_names(self):
        """Returns a list of available class names loaded from the script."""
        return list(self.classes.keys())
    
    def instantiate(self, class_name: str, *args, **kwargs):
        """
        Instantiates the specified class with the provided arguments.
        
        Raises:
            ValueError: If the specified class is not found.
        """
        if class_name not in self.classes:
            raise ValueError(f"Class '{class_name}' not found in the provided script.")
        return self.classes[class_name](*args, **kwargs)
    
    def list_members(self, class_name: str):
        """
        Lists member variables and functions of the specified class.
        
        Returns:
            tuple: (member_variables, member_functions)
        """
        if class_name not in self.classes:
            raise ValueError(f"Class '{class_name}' not found in the provided script.")
        
        cls = self.classes[class_name]
        member_functions = [
            name for name, func in inspect.getmembers(cls, predicate=inspect.isfunction)
            if not name.startswith('__') and not name.startswith('_')
        ]
        
        try:
            sig = inspect.signature(cls.__init__)
            dummy_args = {}
            for param_name, param in sig.parameters.items():
                if param_name == 'self':
                    continue
                dummy_args[param_name] = param.default if param.default is not param.empty else f"dummy_{param_name}"
            instance = cls(**dummy_args)
            member_variables = list(vars(instance).keys())
        except Exception as e:
            member_variables = f"Could not instantiate class to list member variables: {e}"
        
        return member_variables, member_functions
    
    def run_member_function(self, instance, function_name: str, *args, **kwargs):
        """
        Executes a member function on the given instance.
        
        Raises:
            AttributeError: If the specified function does not exist.
        """
        if hasattr(instance, function_name):
            method = getattr(instance, function_name)
            return method(*args, **kwargs)
        else:
            raise AttributeError(f"Method '{function_name}' not found in instance of {type(instance).__name__}")
    
    def run_all_member_functions(self, instance, functions_params: dict = None):
        """
        Executes all member functions of the instance.
        
        Returns:
            dict: Mapping of function names to their outputs.
        """
        outputs = {}
        member_functions = [
            name for name, func in inspect.getmembers(instance, predicate=inspect.ismethod)
            if not name.startswith('__') and not name.startswith('_')
        ]
        for func_name in member_functions:
            args, kwargs = (), {}
            if functions_params and func_name in functions_params:
                args, kwargs = functions_params[func_name]
            try:
                outputs[func_name] = self.run_member_function(instance, func_name, *args, **kwargs)
            except Exception as e:
                outputs[func_name] = f"Error: {e}"
        return outputs
    
    def instantiate_composite(self, *args, **kwargs):
        """
        Creates a composite class inheriting from all loaded classes.
        For classes in an inheritance chain, only the most-derived class is used.
        """
        if not self.classes:
            raise ValueError("No classes loaded from the provided script.")
        
        def get_depth(cls):
            depth = 0
            for base in cls.__mro__:
                if base is object:
                    break
                depth += 1
            return depth
        
        most_derived = max(self.classes.values(), key=get_depth)
        mro_set = set(most_derived.__mro__)
        other_bases = tuple(cls for cls in self.classes.values() 
                            if cls is not most_derived and cls not in mro_set)
        
        if not other_bases:
            Composite = most_derived
        else:
            bases = (most_derived,) + other_bases
            Composite = type("Composite", bases, {})
        
        return Composite(*args, **kwargs)
    
    def get_composite_init_signature(self):
        """
        Returns the signature of the __init__ method used by the composite instantiator.
        """
        if not self.classes:
            raise ValueError("No classes loaded from the provided script.")
        
        def get_depth(cls):
            depth = 0
            for base in cls.__mro__:
                if base is object:
                    break
                depth += 1
            return depth
        
        most_derived = max(self.classes.values(), key=get_depth)
        return inspect.signature(most_derived.__init__)
    
    def instantiate_composite_with_params(self, params: dict):
        """
        Instantiates the composite class using parameters from the provided dictionary.
        
        Raises:
            ValueError: If a required parameter is missing.
        """
        sig = self.get_composite_init_signature()
        required_params = [name for name, param in sig.parameters.items() if name != "self"]
        
        for param in required_params:
            if param not in params:
                raise ValueError(f"Missing required parameter: {param}")
        
        return self.instantiate_composite(**params)

############################################
# BlockData: Stores the Block's Data
############################################

class BlockData:
    def __init__(self, value=None):
        self.value = value if isinstance(value, dict) else {"value": value}

    def __str__(self):
        return f"BlockData({self.value})"

############################################
# Block Class
############################################

class Block:
    def __init__(self, block_id, dir_neighbors, position=None, data=None):
        self.id = block_id  # Integer ID for user reference
        self.data = BlockData(data)
        self.neighbors = set(dir_neighbors)  # Set of neighbor block hashes
        self.position = position if position is not None else (0, 0, 0)
        self.proposer = None
        self.hash = self.compute_hash()  # Compute hash based on content

    def compute_hash(self):
        sha = hashlib.sha256()
        position_str = str(self.position)
        data_str = json.dumps(self.data.value, sort_keys=True)
        neighbors_str = ''.join(sorted(self.neighbors))  # Sort neighbor hashes for consistency
        sha.update(f"{position_str}{data_str}{neighbors_str}".encode('utf-8'))
        return sha.hexdigest()

############################################
# MultiDimensionalBlockchain Class
############################################

class MultiDimensionalBlockchain:
    def __init__(self, instance_script):
        self.blocks = {}  # Maps block hash to Block objects
        self.id_to_hash = {}  # Maps integer ID to block hash
        self.hash_to_id = {}  # Maps block hash to integer ID
        self.next_id = 0  # Counter for new blocks
        self.positions_map = {}  # Maps (x, y, z) to block hash
        self.open_positions = set()  # Positions adjacent to an existing block
        self.instance_script = instance_script
        self.Smart_Instance = Instantiator(instance_script)
        self.proposer = "mdb_auth"
        self.reverse_graph = defaultdict(set)

        # Create genesis block at (0,0,0)
        genesis_position = (0, 0, 0)
        genesis = Block(self.next_id, [], genesis_position, 'Genesis Block')
        genesis_hash = genesis.hash
        self.blocks[genesis_hash] = genesis
        self.id_to_hash[0] = genesis_hash
        self.hash_to_id[genesis_hash] = 0
        self.positions_map[genesis_position] = genesis_hash
        self.next_id += 1

        # Update open positions
        for dx, dy, dz in [(1, 0, 0), (-1, 0, 0), (0, 1, 0), (0, -1, 0), (0, 0, 1), (0, 0, -1)]:
            pos = (genesis_position[0] + dx, genesis_position[1] + dy, genesis_position[2] + dz)
            self.open_positions.add(pos)


    def add_block(self, data, position=None):
        if position is None:
            if self.open_positions:
                position = min(self.open_positions, key=lambda pos: pos[0]**2 + pos[1]**2 + pos[2]**2)
                self.open_positions.remove(position)
            else:
                print("No available open positions to place new block.")
                return None
        else:
            if position in self.positions_map:
                print("Provided position is already occupied.")
                return None
            adjacent_found = any(
                (position[0] + dx, position[1] + dy, position[2] + dz) in self.positions_map
                for dx, dy, dz in [(1, 0, 0), (-1, 0, 0), (0, 1, 0), (0, -1, 0), (0, 0, 1), (0, 0, -1)]
            )
            if not adjacent_found:
                print("Provided position is not adjacent to any existing block.")
                return None
            if position in self.open_positions:
                self.open_positions.remove(position)

        # Find neighbor hashes
        neighbors = []
        for dx, dy, dz in [(1, 0, 0), (-1, 0, 0), (0, 1, 0), (0, -1, 0), (0, 0, 1), (0, 0, -1)]:
            neighbor_pos = (position[0] + dx, position[1] + dy, position[2] + dz)
            if neighbor_pos in self.positions_map:
                neighbor_hash = self.positions_map[neighbor_pos]
                neighbors.append(neighbor_hash)
            else:
                self.open_positions.add(neighbor_pos)

        # Create and store new block
        new_block = Block(self.next_id, neighbors, position, data)
        new_block_hash = new_block.hash
        self.blocks[new_block_hash] = new_block
        self.id_to_hash[self.next_id] = new_block_hash
        self.hash_to_id[new_block_hash] = self.next_id
        self.positions_map[position] = new_block_hash

        # Update reverse graph
        for neighbor_hash in new_block.neighbors:
            self.reverse_graph[neighbor_hash].add(new_block_hash)
        self.next_id += 1
        return new_block

    def get_adjacency_list(self):
        return {
            self.hash_to_id[block_hash]: [self.hash_to_id[neighbor_hash] for neighbor_hash in block.neighbors]
            for block_hash, block in self.blocks.items()
        }

    def validate_blockchain(self):
        for block_hash, block in self.blocks.items():
            if block_hash != block.compute_hash():
                return False
            for neighbor_hash in block.neighbors:
                if neighbor_hash not in self.blocks:
                    return False
        return True

    def get_neighbors(self, block_hash):
        """Get all neighbors of a block, including both outgoing and incoming edges."""
        block = self.blocks[block_hash]
        neighbors = set(block.neighbors)
        incoming = self.reverse_graph[block_hash]
        return neighbors.union(incoming)

    def shortest_path_dijkstra(self, start_hash, end_hash):
        distances = {block_hash: float('inf') for block_hash in self.blocks}
        previous = {block_hash: None for block_hash in self.blocks}
        distances[start_hash] = 0
        visited = set()
        queue = [(0, start_hash)]
        while queue:
            current_distance, current = heapq.heappop(queue)
            if current == end_hash:
                break
            if current in visited:
                continue
            visited.add(current)
            for neighbor in self.blocks[current].neighbors:
                distance = current_distance + 1
                if distance < distances[neighbor]:
                    distances[neighbor] = distance
                    previous[neighbor] = current
                    heapq.heappush(queue, (distance, neighbor))
        path = []
        current = end_hash
        while current is not None:
            path.insert(0, current)
            current = previous[current]
        if path and path[0] != start_hash:
            return None
        return path

    def shortest_path_a_star(self, start_hash, end_hash):
        def heuristic(a_hash, b_hash):
            a = self.blocks[a_hash]
            b = self.blocks[b_hash]
            return math.sqrt((a.position[0] - b.position[0]) ** 2 + (a.position[1] - b.position[1]) ** 2)
        open_set = {start_hash}
        came_from = {}
        g_score = {block_hash: float('inf') for block_hash in self.blocks}
        f_score = {block_hash: float('inf') for block_hash in self.blocks}
        g_score[start_hash] = 0
        f_score[start_hash] = heuristic(start_hash, end_hash)
        open_heap = [(f_score[start_hash], start_hash)]
        while open_heap:
            current_f, current = heapq.heappop(open_heap)
            if current == end_hash:
                path = []
                while current in came_from:
                    path.insert(0, current)
                    current = came_from[current]
                path.insert(0, start_hash)
                return path
            open_set.discard(current)
            for neighbor in self.blocks[current].neighbors:
                tentative_g_score = g_score[current] + 1
                if tentative_g_score < g_score[neighbor]:
                    came_from[neighbor] = current
                    g_score[neighbor] = tentative_g_score
                    f_score[neighbor] = tentative_g_score + heuristic(neighbor, end_hash)
                    if neighbor not in open_set:
                        open_set.add(neighbor)
                        heapq.heappush(open_heap, (f_score[neighbor], neighbor))
        return None

    def find_path(self, start_hash, target_hash):
        """Find the shortest path between start_hash and target_hash using BFS."""
        if start_hash not in self.blocks or target_hash not in self.blocks:
            return None  # Invalid block hashes

        visited = set()
        queue = deque([start_hash])
        parent = {start_hash: None}  # To reconstruct the path

        while queue:
            current_hash = queue.popleft()
            if current_hash == target_hash:
                path = []
                while current_hash is not None:
                    path.append(current_hash)
                    current_hash = parent[current_hash]
                return path[::-1]  # Reverse to get start-to-target order

            if current_hash not in visited:
                visited.add(current_hash)
                for neighbor_hash in self.get_neighbors(current_hash):
                    if neighbor_hash not in visited:
                        queue.append(neighbor_hash)
                        if neighbor_hash not in parent:
                            parent[neighbor_hash] = current_hash
        return None  # No path found

    def print_path(self, start_id, target_id):
        """Print the path between two blocks with their positions using integer IDs."""
        start_hash = self.id_to_hash.get(int(start_id))
        target_hash = self.id_to_hash.get(int(target_id))
        if not start_hash or not target_hash:
            print("Invalid block IDs.")
            return []
        path_hashes = self.find_path(start_hash, target_hash)
        if path_hashes is None:
            print(f"No path exists between Block {start_id} and Block {target_id}.")
            return []
        else:
            path_ids = [self.hash_to_id[hash] for hash in path_hashes]
            print(f"Path from Block {start_id} to Block {target_id}:")
            for block_id in path_ids:
                block = self.blocks[self.id_to_hash[block_id]]
                neighbor_ids = [self.hash_to_id[h] for h in block.neighbors]
                print(f"Block ID: {block_id}, Position: {block.position}, Neighbours: {neighbor_ids}")
            return path_ids



    def get_position(self, block_id):
        return self.blocks[block_id].position

    def euclidean_distance(self, pos1, pos2):
        """Calculate Euclidean distance between two 3D positions."""
        return sqrt(sum((a - b) ** 2 for a, b in zip(pos1, pos2)))

    def find_path2(self, start_id, target_id):
        """Find the shortest path using A* algorithm."""
        if start_id not in self.blocks or target_id not in self.blocks:
            return None

        start_pos = self.get_position(start_id)
        target_pos = self.get_position(target_id)

        def heuristic(block_id):
            return self.euclidean_distance(self.get_position(block_id), target_pos)

        # Priority queue for A*: (f_score, g_score, block_id)
        frontier = []
        heapq.heappush(frontier, (0 + heuristic(start_id), 0, start_id))
        came_from = {start_id: None}  # Tracks the path
        cost_so_far = {start_id: 0}   # Tracks g_score (hops so far)

        while frontier:
            _, current_cost, current_id = heapq.heappop(frontier)

            if current_id == target_id:
                # Reconstruct and return the path
                path = []
                while current_id is not None:
                    path.append(current_id)
                    current_id = came_from[current_id]
                return path[::-1]

            for neighbor_id in self.get_neighbors(current_id):
                new_cost = current_cost + 1  # Uniform cost per hop
                if neighbor_id not in cost_so_far or new_cost < cost_so_far[neighbor_id]:
                    cost_so_far[neighbor_id] = new_cost
                    priority = new_cost + heuristic(neighbor_id)  # f_score = g_score + h_score
                    heapq.heappush(frontier, (priority, new_cost, neighbor_id))
                    came_from[neighbor_id] = current_id

        return None  # No path found

    def get_neighbors(self, block_id):
        """Retrieve all neighbors (outgoing and incoming)."""
        outgoing = self.blocks[block_id].neighbors
        incoming = self.reverse_graph[block_id]
        return outgoing.union(incoming)
    ############################################
    # Smart Instance Management
    ############################################

    def init_block_smart_instance(self,data):
        """
        Initializes a smart instance for a block by prompting for required parameters.
        """
        composite_sig = self.Smart_Instance.get_composite_init_signature()
        params = {}
        for pname, param in composite_sig.parameters.items():
            if pname != "self":
                try:

                    if data.startswith('Simulating'):
                        params[pname] = sim_prompt_for_parameter(pname, param.annotation)
                    else:
                        params[pname] = prompt_for_parameter(pname, param.annotation)
                except:
                    params[pname] = prompt_for_parameter(pname, param.annotation)
    
        composite_instance = self.Smart_Instance.instantiate_composite_with_params(params)
        return composite_instance

    def update_block_smart_instance(self, block_id):
        """
        Allows the user to select and run a smart instance method on a given block.
        """
        instance = self.blocks[block_id].smart_instance
        funcs = [(name, m) for name, m in inspect.getmembers(instance, predicate=inspect.ismethod)
                 if not name.startswith('__') and not name.startswith('_')]
        if not funcs:
            print("No smart instance functions available.")
            return 0
        print("\nSmart Instance Functions:")
        for i, (fname, _) in enumerate(funcs):
            print(f"{i+1}. {fname}")
        choice = input("Run which function? (Enter number, blank to skip): ").strip()
        if not choice:
            return 0
        try:
            idx = int(choice) - 1
            if idx < 0 or idx >= len(funcs):
                print("Invalid choice.")
                return 0
            chosen_func = funcs[idx][1]
        except Exception as e:
            print("Invalid input:", e)
            return 0
        sig = inspect.signature(chosen_func)
        params = {}
        for pname, param in sig.parameters.items():
            if pname != "self":
                params[pname] = prompt_for_parameter(pname, param.annotation)
        try:
            # Use keyword argument unpacking if any parameters are provided.
            result = chosen_func(**params) if params else chosen_func()
            print("Smart instance function result:", result)
            return result
        except Exception as e:
            print("Error running smart instance function:", e)
            return 0




def main():
    # Default instance script for the smart instance.
    DEFAULT_INSTANCE_SCRIPT = """
class BankAccount:
    def __init__(self, name: str, balance: float) -> None:
        self.name: str = name
        self.balance: float = balance
        self.transactions: list[dict[str, float | str]] = []  # List to record transaction logs

    # Private method to record a transaction
    def _create_transaction(self, operation: str, amount: float, new_balance: float) -> None:
        transaction: dict[str, float | str] = {
            "operation": operation,
            "amount": amount,
            "balance": new_balance
        }
        self.transactions.append(transaction)

    # Deposit method: increases balance and logs the transaction
    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Deposit amount must be positive.")
        self.balance += amount
        self._create_transaction("deposit", amount, self.balance)

    # Withdraw method: decreases balance if funds are sufficient and logs the transaction
    def withdraw(self, amount: float) -> bool:
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive.")
        if self.balance >= amount:
            self.balance -= amount
            self._create_transaction("withdraw", amount, self.balance)
            return True
        else:
            print("Insufficient funds for withdrawal.")
            return False

    # Retrieve current balance
    def get_balance(self) -> float:
        return self.balance

    # Display all transaction logs
    def show_transactions(self) -> None:
        print(f"Transaction history for {self.name}:")
        if not self.transactions:
            print("No transactions available.")
        else:
            for txn in self.transactions:
                print(f"{txn['operation'].capitalize()}: {txn['amount']:.2f}, Balance: {txn['balance']:.2f}")

"""

    blockchain = MultiDimensionalBlockchain(DEFAULT_INSTANCE_SCRIPT)

    num_blocks = 1000
    for i in range(num_blocks):
        sim_block = blockchain.add_block(f'Simulating Block Addition at {i+1}')

    menu = """
Please choose an option:
1. Add a new block
2. Print blockchain data
3. Display adjacency list (first N blocks)
4. Validate blockchain
5. Run Smart Instance on a Block
6. Find Shortest Path between Two Blocks
7. Exit
"""
    while True:
        print(menu)
        choice = input("Enter your choice (1-7): ").strip()
        if choice == "1":
            base_data = input("Enter block data (free text): ")
            data = {"info": base_data, "rand": random.randint(100, 999)}
            new_block = blockchain.add_block(data)
            if new_block:
                print(f"Block added with ID: {new_block.id}")
            else:
                print("Block addition failed.")
        elif choice == "2":
            print("Blockchain Data:")
            for bid in sorted(blockchain.id_to_hash.keys()):
                block_hash = blockchain.id_to_hash[bid]
                block = blockchain.blocks[block_hash]
                neighbor_ids = [blockchain.hash_to_id[h] for h in block.neighbors]
                print(f"Block ID: {block.id} | Hash: {block_hash[:8]}... | Data: {vars(block.data)} | Position: {block.position} | Proposer: {block.proposer} | Neighbors: {neighbor_ids}")
            print("")
        elif choice == "3":
            try:
                n = int(input("Enter number of blocks to display (default 10): ").strip() or 10)
            except:
                n = 10
            adj = blockchain.get_adjacency_list()
            print("Adjacency List:")
            for i in range(n):
                print(f"Block {i}: Neighbors -> {adj.get(i, 'Not exist')}")
            print("")
        elif choice == "4":
            valid = blockchain.validate_blockchain()
            print(f"Blockchain validation result: {valid}\n")
        elif choice == "5":
            try:
                bid = int(input("Enter block ID to update its smart instance: ").strip())
                blockchain.update_block_smart_instance(bid)
            except Exception as e:
                print("Invalid block ID:", e)
        elif choice == "6":
            try:
                start = int(input("Enter Starting block ID: ").strip())
                end = int(input("Enter Ending block ID: ").strip())

                tic = time.time()
                path = blockchain.print_path(start, end)
                print(path, f"Hops: {len(path)}")
                print("Time delay for traversal:", time.time() - tic)

                # Uncomment and update other path-finding methods if needed
                # start_hash = blockchain.id_to_hash[start]
                # end_hash = blockchain.id_to_hash[end]
                # tic = time.time()
                # path_hashes = blockchain.shortest_path_dijkstra(start_hash, end_hash)
                # path = [blockchain.hash_to_id[h] for h in path_hashes] if path_hashes else []
                # print(path, f"Hops: {len(path)}")
                # print("Time delay for traversal:", time.time() - tic)
            except Exception as e:
                print("Invalid block ID:", e)
        elif choice == "7":
            print("Exiting. Goodbye!")
            break
        else:
            print("Invalid choice. Try again.\n")

main()


Please choose an option:
1. Add a new block
2. Print blockchain data
3. Display adjacency list (first N blocks)
4. Validate blockchain
5. Run Smart Instance on a Block
6. Find Shortest Path between Two Blocks
7. Exit

Blockchain Data:
Block ID: 0 | Hash: ad5b83f3... | Data: {'value': {'value': 'Genesis Block'}} | Position: (0, 0, 0) | Proposer: None | Neighbors: []
Block ID: 1 | Hash: 835c33a7... | Data: {'value': {'value': 'Simulating Block Addition at 1'}} | Position: (0, 1, 0) | Proposer: None | Neighbors: [0]
Block ID: 2 | Hash: 776f0b09... | Data: {'value': {'value': 'Simulating Block Addition at 2'}} | Position: (0, -1, 0) | Proposer: None | Neighbors: [0]
Block ID: 3 | Hash: b22a4095... | Data: {'value': {'value': 'Simulating Block Addition at 3'}} | Position: (1, 0, 0) | Proposer: None | Neighbors: [0]
Block ID: 4 | Hash: cac68dc1... | Data: {'value': {'value': 'Simulating Block Addition at 4'}} | Position: (-1, 0, 0) | Proposer: None | Neighbors: [0]
Block ID: 5 | Hash: b98e4